In [1]:
import os
import json
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Inisialisasi path
CSV_PATH = "data/processed/cases.csv"
QUERIES_PATH = "data/eval/queries.json"
RESULTS_DIR = "data/results"
os.makedirs(RESULTS_DIR, exist_ok=True)

# Memuat data basis kasus
df = pd.read_csv(CSV_PATH)
with open(QUERIES_PATH, 'r', encoding='utf-8') as f:
    queries = json.load(f)

# Re-fit TF-IDF Vectorizer pada seluruh basis data ringkasan fakta
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(df['ringkasan_fakta'].fillna(''))

def predict_outcome(query_text, k=5):
    # Hitung similarity
    query_vector = tfidf.transform([query_text.lower()])
    similarity_scores = cosine_similarity(query_vector, tfidf_matrix).flatten()
    
    # Ambil Top-K indeks teratas
    top_k_indices = similarity_scores.argsort()[-k:][::-1]
    top_5_ids = [int(df.iloc[i]['case_id']) for i in top_k_indices]
    
    # Ambil solusi (hukuman) dari kasus paling mirip (Rank 1 / Bobot tertinggi)
    predicted_sol = df.iloc[top_k_indices[0]]['hukuman_solusi']
    
    return predicted_sol, str(top_5_ids)

print("✅ Fungsi predict_outcome() siap dijalankan.")

✅ Fungsi predict_outcome() siap dijalankan.


In [2]:
predictions_results = []

for q in queries:
    pred_sol, top_5 = predict_outcome(q['query_text'], k=5)
    predictions_results.append({
        "query_id": q['query_id'],
        "predicted_solution": pred_sol,
        "top_5_case_ids": top_5
    })

# Simpan ke format file sesuai ketentuan modul (/data/results/predictions.csv)
df_pred = pd.DataFrame(predictions_results)
output_pred_path = os.path.join(RESULTS_DIR, "predictions.csv")
df_pred.to_csv(output_pred_path, index=False, encoding='utf-8')

print(f"🎉 Sukses! File prediksi berhasil dibuat di: {output_pred_path}")
df_pred.head()

🎉 Sukses! File prediksi berhasil dibuat di: data/results\predictions.csv


,query_id,predicted_solution,top_5_case_ids
0,18,2 dua tahun dan 6 enam bulan penjara dengan pe...,"[18, 1, 37, 21, 31]"
1,14,8 delapan tahun dengan perintah terdakwa ditah...,"[14, 13, 10, 27, 8]"
2,5,1 satu tahun dan membayar denda sebesar rp,"[5, 1, 37, 25, 31]"
3,30,6 enam bulan dan denda sebesar rp50,"[30, 4, 12, 31, 25]"
4,36,2 dua tahun dikurangi selama terdakwa berada d...,"[36, 2, 6, 21, 37]"
